# Kinship Library
## Notebook 04: SQL, Final Data Preparation and Streamlit Assets

### Purpose
Prepare the completed NLP results for the application layer without retraining the NLP model.

### Inputs
`data/processed/books_model_ready.csv`

`models/book_embeddings.npy`

### Outputs
`data/kinship_library.db`

`data/processed/app_books.csv`

`models/app_book_embeddings.npy`

`sql/schema.sql`

`sql/analysis_queries.sql`

## 1. Setup

### Purpose
Mount Google Drive and define permanent project paths.

In [ ]:
from google.colab import drive
from pathlib import Path
import sqlite3
import numpy as np
import pandas as pd

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/kinship_library")
DATA = PROJECT_ROOT / "data"
PROCESSED_DATA = DATA / "processed"
MODELS = PROJECT_ROOT / "models"
SQL_FOLDER = PROJECT_ROOT / "sql"
SQL_FOLDER.mkdir(parents=True, exist_ok=True)

MODEL_READY_FILE = PROCESSED_DATA / "books_model_ready.csv"
EMBEDDINGS_FILE = MODELS / "book_embeddings.npy"
DATABASE_FILE = DATA / "kinship_library.db"
APP_BOOKS_FILE = PROCESSED_DATA / "app_books.csv"
APP_EMBEDDINGS_FILE = MODELS / "app_book_embeddings.npy"
SCHEMA_FILE = SQL_FOLDER / "schema.sql"
QUERIES_FILE = SQL_FOLDER / "analysis_queries.sql"

print("Model data exists:", MODEL_READY_FILE.exists())
print("Embeddings exist:", EMBEDDINGS_FILE.exists())

Mounted at /content/drive
Model data exists: True
Embeddings exist: True


## 2. Load and validate the final modeling outputs

### Expected result
There should be 713 NLP eligible books and one embedding row per book.

In [ ]:
model_books = pd.read_csv(MODEL_READY_FILE)
book_embeddings = np.load(EMBEDDINGS_FILE)

print("Model books:", len(model_books))
print("Embedding matrix:", book_embeddings.shape)

assert len(model_books) == len(book_embeddings)
display(model_books.head())

Model books: 713
Embedding matrix: (713, 384)


,source_id,title,author,description,subjects,published_year,average_rating,ratings_count,isbn_x,language,...,use_google_metadata,isbn_y,description_final,semantic_text,semantic_text_words,nlp_text_quality,tfidf_text,cluster,pca_1,pca_2
0,/works/OL6772557W,The ecology of human development,Urie Bronfenbrenner,NaN,"Child psychology, Research, Human Development,...",1979.0,5.000000,1.0,9780674224575,eng,...,True,9780674224575,"To understand the way children develop, Bronfe...",The ecology of human development. Urie Bronfen...,66,rich,the ecology of human development urie bronfenb...,0,-0.230222,0.275399
1,/works/OL3458484W,People and nature,Emilio F. Moran,NaN,"Effect of environment on, Effect of environmen...",2006.0,NaN,NaN,1118877411,eng,...,False,1118877411,NaN,People and nature. Emilio F. Moran. Effect of ...,49,rich,people and nature emilio f moran effect of env...,0,-0.311980,0.212824
2,/works/OL2990533W,The green imperative,"Victor J. Papanek, Victor Papanek",NaN,"Design, industrial, Environmental aspects, Env...",1995.0,NaN,NaN,9780500296196,eng,...,True,9780500296196,A fresh edition of the sustainable design pion...,"The green imperative. Victor J. Papanek, Victo...",240,rich,the green imperative victor j papanek victor p...,0,-0.095519,0.303913
3,/works/OL1805267W,Design with nature,Ian L. McHarg,NaN,"Effect of environment on, Effect of human bein...",1969.0,4.333334,6.0,047111460X,eng,...,True,047111460X,"This is a new paperback version of a text, ori...",Design with nature. Ian L. McHarg. Effect of e...,78,rich,design with nature ian l mcharg effect of envi...,0,-0.205564,0.307577
4,/works/OL2911248W,"The death of nature: women, ecology, and the s...",Carolyn Merchant,NaN,"Human ecology, Philosophy of nature, Social as...",1980.0,NaN,NaN,9780062505712,eng,...,False,9780062505712,NaN,"The death of nature: women, ecology, and the s...",55,rich,the death of nature women ecology and the scie...,0,-0.296908,0.224371


## 3. Inspect the three semantic clusters

### Purpose
Review the K Means clusters before storing them in SQL. Cluster numbers are identifiers, not intrinsic category names.

In [ ]:
cluster_summary = (
    model_books.groupby("cluster")
    .agg(
        books=("title", "size"),
        themes=("collection_theme", "nunique"),
        mean_text_words=("semantic_text_words", "mean")
    )
    .round(1)
)

display(cluster_summary)
display(pd.crosstab(model_books["cluster"], model_books["collection_theme"]))

,books,themes,mean_text_words
cluster,,,
0,310,5,115.5
1,229,5,112.9
2,174,4,51.8


collection_theme,ayurveda_indian_spirituality,humans_and_nature,indigenous_knowledge_americas,yoga_philosophy,yoga_way_of_life
cluster,,,,,
0,3,146,159,1,1
1,72,2,9,84,62
2,5,1,167,1,0


## 4. Inspect example books from each cluster

### Purpose
Use actual titles to help us interpret the clusters.

In [ ]:
for cluster_id in sorted(model_books["cluster"].unique()):
    print()
    print("=" * 60)
    print("CLUSTER", cluster_id)
    print("=" * 60)

    display(
        model_books[
            model_books["cluster"] == cluster_id
        ][["title", "author", "collection_theme"]].head(12)
    )


CLUSTER 0


,title,author,collection_theme
0,The ecology of human development,Urie Bronfenbrenner,humans_and_nature
1,People and nature,Emilio F. Moran,humans_and_nature
2,The green imperative,"Victor J. Papanek, Victor Papanek",humans_and_nature
3,Design with nature,Ian L. McHarg,humans_and_nature
4,"The death of nature: women, ecology, and the s...",Carolyn Merchant,humans_and_nature
5,Human geography,"Jerome Donald Fellmann, Arthur Getis, Judith G...",humans_and_nature
6,Galen's Prophecy,Jerome Kagan,humans_and_nature
7,Man's responsibility for nature,John Arthur Passmore,humans_and_nature
8,Ecological Transition,John W. Bennett,humans_and_nature
9,Biomimicry,Janine M. Benyus,humans_and_nature



CLUSTER 1


,title,author,collection_theme
120,"Dark night, early dawn",Christopher M. Bache,humans_and_nature
134,Wittgenstein,P. M. S. Hacker,humans_and_nature
149,Yogasūtra,Patañjali.,yoga_philosophy
150,Light on the Yoga Sutras of Patanjali,B. K. S. Iyengar,yoga_philosophy
151,Patanjali Yoga Sutras,Swami Prabhavananda,yoga_philosophy
152,Yoga-Sutra,Kalashatra Govinda,yoga_philosophy
153,The Yoga Sutras of Patanjali,Patanjali,yoga_philosophy
154,The yoga sutras of Patanjali,Satchidananda Swami.,yoga_philosophy
155,How to know God,Patañjali.,yoga_philosophy
156,Yoga Sutras of Patanjali,Charles Johnston,yoga_philosophy



CLUSTER 2


,title,author,collection_theme
60,No more than human,Maura Laverty,humans_and_nature
186,The Upanishads. Tr. by F.M. Müller,Upanishads,yoga_philosophy
365,Tantra,Bernard Soulié,ayurveda_indian_spirituality
369,"Sir John Woodroffe, Tantra and Bengal",Kathleen Taylor,ayurveda_indian_spirituality
371,The dawn of Indian music in the West,Peter Lavezzoli,ayurveda_indian_spirituality
373,The Niśvāsatattvasaṃhitā,"Dominic Goodall, Alexis Sanderson, H. Isaacson",ayurveda_indian_spirituality
375,The Tantras and their impact on Indian life,Pushpendra Kumar,ayurveda_indian_spirituality
430,People of the Himalayas,Kanak Chandra Mahanta,indigenous_knowledge_americas
431,Tribal's and social inclusion,"Suchismita Mishra, D. C. Nanjunda",indigenous_knowledge_americas
434,Report of the Royal Commission on Aboriginal P...,Royal Commission on Aboriginal Peoples (RCAP),indigenous_knowledge_americas


# Part A: Relational database

## 5. Prepare the books table

`collection_theme` remains the search route through which the book entered the catalogue. It is not treated as a definitive genre.

In [ ]:
books_table = model_books.copy()
books_table.insert(0, "book_id", range(1, len(books_table) + 1))

book_columns = [
    "book_id", "source_id", "title", "author", "subjects",
    "published_year", "isbn", "language", "average_rating",
    "ratings_count", "collection_theme", "search_query",
    "description_final", "google_categories", "semantic_text",
    "semantic_text_words", "nlp_text_quality", "cluster",
    "pca_1", "pca_2"
]

book_columns = [c for c in book_columns if c in books_table.columns]
books_table = books_table[book_columns].copy()

print("Books table rows:", len(books_table))
display(books_table.head())

Books table rows: 713


,book_id,source_id,title,author,subjects,published_year,language,average_rating,ratings_count,collection_theme,search_query,description_final,google_categories,semantic_text,semantic_text_words,nlp_text_quality,cluster,pca_1,pca_2
0,1,/works/OL6772557W,The ecology of human development,Urie Bronfenbrenner,"Child psychology, Research, Human Development,...",1979.0,eng,5.000000,1.0,humans_and_nature,ecology human nature,"To understand the way children develop, Bronfe...",Family & Relationships,The ecology of human development. Urie Bronfen...,66,rich,0,-0.230222,0.275399
1,2,/works/OL3458484W,People and nature,Emilio F. Moran,"Effect of environment on, Effect of environmen...",2006.0,eng,NaN,NaN,humans_and_nature,ecology human nature,NaN,NaN,People and nature. Emilio F. Moran. Effect of ...,49,rich,0,-0.311980,0.212824
2,3,/works/OL2990533W,The green imperative,"Victor J. Papanek, Victor Papanek","Design, industrial, Environmental aspects, Env...",1995.0,eng,NaN,NaN,humans_and_nature,ecology human nature,A fresh edition of the sustainable design pion...,Architecture,"The green imperative. Victor J. Papanek, Victo...",240,rich,0,-0.095519,0.303913
3,4,/works/OL1805267W,Design with nature,Ian L. McHarg,"Effect of environment on, Effect of human bein...",1969.0,eng,4.333334,6.0,humans_and_nature,ecology human nature,"This is a new paperback version of a text, ori...",Architecture,Design with nature. Ian L. McHarg. Effect of e...,78,rich,0,-0.205564,0.307577
4,5,/works/OL2911248W,"The death of nature: women, ecology, and the s...",Carolyn Merchant,"Human ecology, Philosophy of nature, Social as...",1980.0,eng,NaN,NaN,humans_and_nature,ecology human nature,NaN,Philosophy of nature,"The death of nature: women, ecology, and the s...",55,rich,0,-0.296908,0.224371


## 6. Normalize authors

### Purpose
Demonstrate a many to many relationship between books and authors.

In [ ]:
author_rows = []

for _, row in books_table.iterrows():
    if pd.isna(row["author"]):
        continue

    for author in str(row["author"]).split(","):
        author = author.strip()
        if author:
            author_rows.append({
                "book_id": row["book_id"],
                "author_name": author
            })

book_author_names = pd.DataFrame(author_rows)

authors_table = (
    book_author_names[["author_name"]]
    .drop_duplicates()
    .sort_values("author_name")
    .reset_index(drop=True)
)
authors_table.insert(0, "author_id", range(1, len(authors_table) + 1))

book_authors = book_author_names.merge(
    authors_table,
    on="author_name",
    how="left"
)[["book_id", "author_id"]].drop_duplicates()

print("Authors:", len(authors_table))
print("Book author relationships:", len(book_authors))

Authors: 881
Book author relationships: 991


## 7. Normalize subjects

### Purpose
Represent the many to many relationship between books and subjects.

In [ ]:
subject_rows = []

for _, row in books_table.iterrows():
    if "subjects" not in books_table.columns or pd.isna(row.get("subjects")):
        continue

    for subject in str(row["subjects"]).split(","):
        subject = subject.strip()
        if subject:
            subject_rows.append({
                "book_id": row["book_id"],
                "subject_name": subject
            })

book_subject_names = pd.DataFrame(subject_rows)

subjects_table = (
    book_subject_names[["subject_name"]]
    .drop_duplicates()
    .sort_values("subject_name")
    .reset_index(drop=True)
)
subjects_table.insert(0, "subject_id", range(1, len(subjects_table) + 1))

book_subjects = book_subject_names.merge(
    subjects_table,
    on="subject_name",
    how="left"
)[["book_id", "subject_id"]].drop_duplicates()

print("Subjects:", len(subjects_table))
print("Book subject relationships:", len(book_subjects))

Subjects: 1902
Book subject relationships: 4594


## 8. Create the SQLite database

In [ ]:
connection = sqlite3.connect(DATABASE_FILE)

books_table.to_sql("books", connection, if_exists="replace", index=False)
authors_table.to_sql("authors", connection, if_exists="replace", index=False)
book_authors.to_sql("book_authors", connection, if_exists="replace", index=False)
subjects_table.to_sql("subjects", connection, if_exists="replace", index=False)
book_subjects.to_sql("book_subjects", connection, if_exists="replace", index=False)

connection.commit()

print("Database saved:", DATABASE_FILE)

Database saved: /content/drive/MyDrive/kinship_library/data/kinship_library.db


## 9. Save the ERD schema for GitHub

### Relationships

`books` many to many `authors` through `book_authors`

`books` many to many `subjects` through `book_subjects`

In [ ]:
schema_sql = '''
CREATE TABLE books (
    book_id INTEGER PRIMARY KEY,
    source_id TEXT,
    title TEXT,
    author TEXT,
    subjects TEXT,
    published_year REAL,
    isbn TEXT,
    language TEXT,
    average_rating REAL,
    ratings_count REAL,
    collection_theme TEXT,
    search_query TEXT,
    description_final TEXT,
    google_categories TEXT,
    semantic_text TEXT,
    semantic_text_words INTEGER,
    nlp_text_quality TEXT,
    cluster INTEGER,
    pca_1 REAL,
    pca_2 REAL
);

CREATE TABLE authors (
    author_id INTEGER PRIMARY KEY,
    author_name TEXT
);

CREATE TABLE book_authors (
    book_id INTEGER,
    author_id INTEGER,
    FOREIGN KEY (book_id) REFERENCES books(book_id),
    FOREIGN KEY (author_id) REFERENCES authors(author_id)
);

CREATE TABLE subjects (
    subject_id INTEGER PRIMARY KEY,
    subject_name TEXT
);

CREATE TABLE book_subjects (
    book_id INTEGER,
    subject_id INTEGER,
    FOREIGN KEY (book_id) REFERENCES books(book_id),
    FOREIGN KEY (subject_id) REFERENCES subjects(subject_id)
);
'''

SCHEMA_FILE.write_text(schema_sql.strip(), encoding="utf-8")
print("Schema saved:", SCHEMA_FILE)

Schema saved: /content/drive/MyDrive/kinship_library/sql/schema.sql


# Part B: SQL analysis

## 10. Books by collection theme

In [ ]:
query_theme = '''
SELECT
    collection_theme,
    COUNT(*) AS number_of_books
FROM books
GROUP BY collection_theme
ORDER BY number_of_books DESC;
'''

display(pd.read_sql_query(query_theme, connection))

,collection_theme,number_of_books
0,indigenous_knowledge_americas,335
1,humans_and_nature,149
2,yoga_philosophy,86
3,ayurveda_indian_spirituality,80
4,yoga_way_of_life,63


## 11. Books by semantic cluster

In [ ]:
query_clusters = '''
SELECT
    cluster,
    COUNT(*) AS number_of_books,
    ROUND(AVG(semantic_text_words), 1) AS avg_text_words
FROM books
GROUP BY cluster
ORDER BY cluster;
'''

display(pd.read_sql_query(query_clusters, connection))

,cluster,number_of_books,avg_text_words
0,0,310,115.5
1,1,229,112.9
2,2,174,51.8


## 12. Most represented authors

### Purpose
Demonstrate a SQL join using the normalized author tables.

In [ ]:
query_authors = '''
SELECT
    a.author_name,
    COUNT(DISTINCT ba.book_id) AS number_of_books
FROM authors AS a
JOIN book_authors AS ba
    ON a.author_id = ba.author_id
GROUP BY a.author_id, a.author_name
ORDER BY number_of_books DESC, a.author_name
LIMIT 20;
'''

display(pd.read_sql_query(query_authors, connection))

,author_name,number_of_books
0,Luis Ochoa Siguencia,7
1,Acharya Prashant,6
2,Swami Vivekananda,6
3,Anonymous,5
4,Gerardo Reichel-Dolmatoff,4
5,Paramhansa Yogananda,4
6,Swami Prabhavananda,4
7,Alan Bicker,3
8,B. K. S. Iyengar,3
9,Creative Publishing Press,3


## 13. Most represented subjects

In [ ]:
query_subjects = '''
SELECT
    s.subject_name,
    COUNT(DISTINCT bs.book_id) AS number_of_books
FROM subjects AS s
JOIN book_subjects AS bs
    ON s.subject_id = bs.subject_id
GROUP BY s.subject_id, s.subject_name
ORDER BY number_of_books DESC, s.subject_name
LIMIT 25;
'''

display(pd.read_sql_query(query_subjects, connection))

,subject_name,number_of_books
0,Indigenous peoples,82
1,Human ecology,73
2,Ecology,62
3,Traditional ecological knowledge,57
4,Indians of South America,56
5,History,46
6,Social life and customs,44
7,Nature,42
8,Religion,40
9,Philosophy,38


## 14. Compare collection themes inside semantic clusters

In [ ]:
query_cluster_theme = '''
SELECT
    cluster,
    collection_theme,
    COUNT(*) AS number_of_books
FROM books
GROUP BY cluster, collection_theme
ORDER BY cluster, number_of_books DESC;
'''

display(pd.read_sql_query(query_cluster_theme, connection))

,cluster,collection_theme,number_of_books
0,0,indigenous_knowledge_americas,159
1,0,humans_and_nature,146
2,0,ayurveda_indian_spirituality,3
3,0,yoga_way_of_life,1
4,0,yoga_philosophy,1
5,1,yoga_philosophy,84
6,1,ayurveda_indian_spirituality,72
7,1,yoga_way_of_life,62
8,1,indigenous_knowledge_americas,9
9,1,humans_and_nature,2


## 15. Save SQL analysis queries

In [ ]:
analysis_queries = (
    "-- Books by collection theme\n"
    + query_theme.strip()
    + "\n\n-- Books by semantic cluster\n"
    + query_clusters.strip()
    + "\n\n-- Most represented authors\n"
    + query_authors.strip()
    + "\n\n-- Most represented subjects\n"
    + query_subjects.strip()
    + "\n\n-- Collection themes inside semantic clusters\n"
    + query_cluster_theme.strip()
)

QUERIES_FILE.write_text(analysis_queries, encoding="utf-8")
print("Queries saved:", QUERIES_FILE)

Queries saved: /content/drive/MyDrive/kinship_library/sql/analysis_queries.sql


# Part C: Streamlit preparation

## 16. Prepare application files

### Purpose
Create compact files for Streamlit while preserving exact row alignment with the embedding matrix.

In [ ]:
app_columns = [
    "book_id", "source_id", "title", "author",
    "collection_theme", "description_final",
    "google_categories", "published_year",
    "average_rating", "ratings_count",
    "cluster", "semantic_text"
]

app_columns = [c for c in app_columns if c in books_table.columns]
app_books = books_table[app_columns].copy()

assert len(app_books) == len(book_embeddings)

app_books.to_csv(APP_BOOKS_FILE, index=False)
np.save(APP_EMBEDDINGS_FILE, book_embeddings)

print("App books:", len(app_books))
print("Embedding rows:", len(book_embeddings))
print("App catalogue saved:", APP_BOOKS_FILE.exists())
print("App embeddings saved:", APP_EMBEDDINGS_FILE.exists())

App books: 713
Embedding rows: 713
App catalogue saved: True
App embeddings saved: True


## 17. Close SQL and final checkpoint

In [ ]:
connection.close()

print("KINSHIP LIBRARY APPLICATION CHECKPOINT")
print()
print("Books prepared for app:", len(app_books))
print("Embedding dimensions:", book_embeddings.shape[1])
print("Semantic clusters:", app_books["cluster"].nunique())
print("SQLite database:", DATABASE_FILE.exists())
print("App catalogue:", APP_BOOKS_FILE.exists())
print("App embeddings:", APP_EMBEDDINGS_FILE.exists())
print("SQL schema:", SCHEMA_FILE.exists())
print("SQL queries:", QUERIES_FILE.exists())
print()
print("The data layer is ready for Streamlit.")

KINSHIP LIBRARY APPLICATION CHECKPOINT

Books prepared for app: 713
Embedding dimensions: 384
Semantic clusters: 3
SQLite database: True
App catalogue: True
App embeddings: True
SQL schema: True
SQL queries: True

The data layer is ready for Streamlit.


# 18. Cover Image Enrichment

The recommendation model is already complete. This section adds book cover information only for the Streamlit interface.

The cover enrichment does **not** change the 713 books, their order, their clusters, or their embeddings.

We first use ISBN values already stored in the catalogue to create Open Library Covers API URLs. If an ISBN is unavailable, the app will later use a Kinship Library placeholder image.

This keeps the presentation layer separate from the NLP model and avoids making hundreds of new Google Books requests.


## 18.1 Inspect ISBN availability

**Purpose**

Measure how many app books already have an ISBN that can be used to request a cover.

**Expected result**

The number of rows remains exactly the same as the application catalogue.


In [21]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/content/drive/MyDrive/kinship_library")

GOOGLE_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "google_books_enrichment.csv"
)

google_books = pd.read_csv(GOOGLE_FILE)

print("Rows:", len(google_books))
print()
print("Columns:")
print(google_books.columns.tolist())

Rows: 920

Columns:
['status', 'google_books_id', 'google_title', 'google_authors', 'description', 'categories', 'publisher', 'google_published_date', 'google_average_rating', 'google_ratings_count', 'source_id', 'open_library_title', 'open_library_author', 'isbn']


In [22]:
import re
import numpy as np
import pandas as pd
from pathlib import Path
from google.colab import drive

# Connect Google Drive
drive.mount("/content/drive")

# Project paths
PROJECT_ROOT = Path("/content/drive/MyDrive/kinship_library")

PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"
RAW_DATA = PROJECT_ROOT / "data" / "raw"
MODELS = PROJECT_ROOT / "models"

APP_BOOKS_FILE = PROCESSED_DATA / "app_books.csv"
APP_EMBEDDINGS_FILE = MODELS / "app_book_embeddings.npy"
GOOGLE_FILE = RAW_DATA / "google_books_enrichment.csv"


# 1. Load final app data
app_books = pd.read_csv(APP_BOOKS_FILE)
book_embeddings = np.load(APP_EMBEDDINGS_FILE)

print("Books loaded:", len(app_books))
print("Embeddings shape:", book_embeddings.shape)

assert len(app_books) == book_embeddings.shape[0], (
    "The catalogue and embeddings are not aligned."
)


# 2. Load previous Google Books enrichment
google_books = pd.read_csv(GOOGLE_FILE)

print()
print("Google Books enrichment rows:", len(google_books))
print("ISBN available:", "isbn" in google_books.columns)


# 3. Create ISBN lookup using source_id
isbn_lookup = (
    google_books[
        ["source_id", "isbn"]
    ]
    .dropna(subset=["source_id"])
    .drop_duplicates(subset="source_id")
)


# 4. Add ISBN to final app catalogue
app_books_with_covers = app_books.merge(
    isbn_lookup,
    on="source_id",
    how="left"
)

assert len(app_books_with_covers) == len(app_books), (
    "The number of books changed during the ISBN merge."
)

print("Books after ISBN merge:", len(app_books_with_covers))


# 5. Clean ISBN
def clean_isbn(value):

    if pd.isna(value):
        return np.nan

    text = str(value).strip()

    text = re.sub(
        r"[^0-9Xx]",
        "",
        text
    )

    if len(text) in [10, 13]:
        return text.upper()

    return np.nan


app_books_with_covers["isbn_clean"] = (
    app_books_with_covers["isbn"]
    .apply(clean_isbn)
)


# 6. Measure ISBN coverage
isbn_count = (
    app_books_with_covers["isbn_clean"]
    .notna()
    .sum()
)

isbn_pct = (
    isbn_count
    / len(app_books_with_covers)
    * 100
)

print()
print("Books in app catalogue:", len(app_books_with_covers))
print("Books with usable ISBN:", isbn_count)
print("ISBN coverage:", round(isbn_pct, 2), "%")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Books loaded: 713
Embeddings shape: (713, 384)

Google Books enrichment rows: 920
ISBN available: True
Books after ISBN merge: 713

Books in app catalogue: 713
Books with usable ISBN: 644
ISBN coverage: 90.32 %


## 18.2 Create Open Library cover URLs

Open Library provides cover images using ISBN identifiers. We create the URL from the ISBN already stored in the catalogue.

The parameter `default=false` is important because Open Library then returns a missing response instead of a generic image when a cover is unavailable. Streamlit can catch that case and show our own Kinship Library placeholder.


In [23]:
def open_library_cover_url(isbn):

    if pd.isna(isbn):
        return np.nan

    return (
        f"https://covers.openlibrary.org/"
        f"b/isbn/{isbn}-L.jpg?default=false"
    )


app_books_with_covers["cover_url"] = (
    app_books_with_covers["isbn_clean"]
    .apply(open_library_cover_url)
)

print(
    "Books with potential cover URL:",
    app_books_with_covers["cover_url"].notna().sum()
)

display(
    app_books_with_covers[
        [
            "title",
            "author",
            "isbn_clean",
            "cover_url"
        ]
    ].head(10)
)

Books with potential cover URL: 644


,title,author,isbn_clean,cover_url
0,The ecology of human development,Urie Bronfenbrenner,9780674224575,https://covers.openlibrary.org/b/isbn/97806742...
1,People and nature,Emilio F. Moran,1118877411,https://covers.openlibrary.org/b/isbn/11188774...
2,The green imperative,"Victor J. Papanek, Victor Papanek",9780500296196,https://covers.openlibrary.org/b/isbn/97805002...
3,Design with nature,Ian L. McHarg,047111460X,https://covers.openlibrary.org/b/isbn/04711146...
4,"The death of nature: women, ecology, and the s...",Carolyn Merchant,9780062505712,https://covers.openlibrary.org/b/isbn/97800625...
5,Human geography,"Jerome Donald Fellmann, Arthur Getis, Judith G...",9780697123244,https://covers.openlibrary.org/b/isbn/97806971...
6,Galen's Prophecy,Jerome Kagan,185343390X,https://covers.openlibrary.org/b/isbn/18534339...
7,Man's responsibility for nature,John Arthur Passmore,0715608193,https://covers.openlibrary.org/b/isbn/07156081...
8,Ecological Transition,John W. Bennett,9781351304702,https://covers.openlibrary.org/b/isbn/97813513...
9,Biomimicry,Janine M. Benyus,0688136915,https://covers.openlibrary.org/b/isbn/06881369...


## 18.3 Optional visual cover check

This cell displays a small sample of cover links. It does not download or store the images.

A valid ISBN does not guarantee that Open Library has a cover, so the Streamlit app will still include a local fallback image.


In [24]:
cover_sample = app_books_with_covers[
    app_books_with_covers["cover_url"].notna()
][["title", "author", "cover_url"]].head(12)

display(cover_sample)


,title,author,cover_url
0,The ecology of human development,Urie Bronfenbrenner,https://covers.openlibrary.org/b/isbn/97806742...
1,People and nature,Emilio F. Moran,https://covers.openlibrary.org/b/isbn/11188774...
2,The green imperative,"Victor J. Papanek, Victor Papanek",https://covers.openlibrary.org/b/isbn/97805002...
3,Design with nature,Ian L. McHarg,https://covers.openlibrary.org/b/isbn/04711146...
4,"The death of nature: women, ecology, and the s...",Carolyn Merchant,https://covers.openlibrary.org/b/isbn/97800625...
5,Human geography,"Jerome Donald Fellmann, Arthur Getis, Judith G...",https://covers.openlibrary.org/b/isbn/97806971...
6,Galen's Prophecy,Jerome Kagan,https://covers.openlibrary.org/b/isbn/18534339...
7,Man's responsibility for nature,John Arthur Passmore,https://covers.openlibrary.org/b/isbn/07156081...
8,Ecological Transition,John W. Bennett,https://covers.openlibrary.org/b/isbn/97813513...
9,Biomimicry,Janine M. Benyus,https://covers.openlibrary.org/b/isbn/06881369...


## 18.4 Save the final Streamlit catalogue

We now update `app_books.csv` with `isbn_clean` and `cover_url`.

Before saving, we verify that the number and order of books have not changed. This is essential because each row must remain aligned with the corresponding row in `app_book_embeddings.npy`.


In [25]:
original_source_ids = app_books["source_id"].astype(str).tolist()
new_source_ids = app_books_with_covers["source_id"].astype(str).tolist()

assert len(app_books_with_covers) == book_embeddings.shape[0], (
    "Catalogue and embedding row counts do not match."
)

assert original_source_ids == new_source_ids, (
    "Book order changed. Do not save because embeddings would become misaligned."
)

app_books_with_covers.to_csv(APP_BOOKS_FILE, index=False)

print("Updated Streamlit catalogue saved to:")
print(APP_BOOKS_FILE)
print()
print("Books:", len(app_books_with_covers))
print("Embedding rows:", book_embeddings.shape[0])
print("Rows with a cover URL:", app_books_with_covers["cover_url"].notna().sum())
print()
print("The recommendation model and embeddings were not changed.")


Updated Streamlit catalogue saved to:
/content/drive/MyDrive/kinship_library/data/processed/app_books.csv

Books: 713
Embedding rows: 713
Rows with a cover URL: 644

The recommendation model and embeddings were not changed.


## 18.5 Final application checkpoint

At this point the data layer is ready for the visual Streamlit application.

The app can use `cover_url` when available and display a custom Kinship Library botanical placeholder when a cover cannot be loaded.


In [26]:
print("KINSHIP LIBRARY STREAMLIT READY")
print()
print("Books:", len(app_books_with_covers))
print("Embedding dimensions:", book_embeddings.shape[1])
print("Semantic clusters:", app_books_with_covers["cluster"].nunique())
print("Cover URL column:", "cover_url" in app_books_with_covers.columns)
print("Catalogue aligned with embeddings:", len(app_books_with_covers) == book_embeddings.shape[0])
print()
print("Next step: build the Streamlit interface.")


KINSHIP LIBRARY STREAMLIT READY

Books: 713
Embedding dimensions: 384
Semantic clusters: 3
Cover URL column: True
Catalogue aligned with embeddings: True

Next step: build the Streamlit interface.
